
# Two-circle split via a Gaussian-mixture $h$-transform

This notebook gives a robust replacement for the two-bump cylinder target from Example 2.

The issue with the original cylinder choice
\[
 g_{\text{cyl}}(\mu) = \exp\Big(-\frac{\lambda}{2}\big\|\langle \Phi,\mu\rangle-a\big\|^2\Big)
\]
is that it only constrains **two smooth moments**. Matching those moments does **not** force every particle into the two circles.

Instead, this notebook uses a positive terminal function that is still a valid Doob $h$-transform but is much sharper:
\[
 g(x) = \sum_{r=1}^R \alpha_r \exp\Big(-\frac{\lambda}{2}\sum_{i=1}^n s_i\,\lVert x_i-y_{r,i}\rVert^2\Big).
\]

Each component is a Gaussian target of the type treated explicitly by Algorithm 1, and the finite sum keeps the construction inside the same $h$-transform framework. The sum over components removes the need to pre-assign labels to left vs. right.

For the six-particle example below we take **all 20 assignments with exactly 3 particles sent left and 3 sent right**.


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from gaussian_mixture_htransform import (
    simulate_gaussian_mixture_terminal_em,
    build_fixed_count_two_cluster_targets,
)

np.set_printoptions(precision=4, suppress=True)


In [ ]:

masses = np.array([0.26, 0.20, 0.17, 0.14, 0.13, 0.10], dtype=float)
initial_positions = np.array([
    [0.50, 0.46],
    [0.52, 0.50],
    [0.48, 0.54],
    [0.54, 0.46],
    [0.46, 0.50],
    [0.50, 0.54],
], dtype=float)

left_center = np.array([0.30, 0.70], dtype=float)
right_center = np.array([0.70, 0.70], dtype=float)
circle_radius = 0.14

# All 20 ways to send exactly 3 of the 6 labels to the left circle.
target_configurations = build_fixed_count_two_cluster_targets(
    num_particles=len(masses),
    left_center=left_center,
    right_center=right_center,
    left_count=3,
)
component_weights = np.ones(len(target_configurations), dtype=float) / len(target_configurations)

lambda_ = 40000.0
horizon = 0.05
step_size = 1.0e-4

print('number of mixture components:', len(target_configurations))
print('lambda =', lambda_)
print('horizon =', horizon)
print('step_size =', step_size)


In [ ]:

def final_circle_stats(final_positions, left_center, right_center, radius):
    centers = np.array([left_center, right_center], dtype=float)
    disp = final_positions[:, None, :] - centers[None, :, :]
    disp = np.mod(disp + 0.5, 1.0) - 0.5
    distances = np.sqrt(np.sum(disp ** 2, axis=-1))
    inside = distances <= radius
    return {
        'inside_each_particle': inside.any(axis=1),
        'left_count': int(inside[:, 0].sum()),
        'right_count': int(inside[:, 1].sum()),
        'all_inside_union': bool(inside.any(axis=1).all()),
        'max_distance_to_nearest_circle': float(np.min(distances, axis=1).max()),
    }

seed = 7
rng = np.random.default_rng(seed)

sim = simulate_gaussian_mixture_terminal_em(
    masses=masses,
    target_configurations=target_configurations,
    component_weights=component_weights,
    lambda_=lambda_,
    horizon=horizon,
    step_size=step_size,
    initial_positions=initial_positions,
    rng=rng,
    store_drifts=True,
)

stats = final_circle_stats(sim.positions[-1], left_center, right_center, circle_radius)
print('seed:', seed)
print('final positions:', sim.positions[-1])
print('stats:', stats)


In [ ]:

def draw_circle(ax, center, radius, **kwargs):
    theta = np.linspace(0.0, 2.0 * np.pi, 256)
    ax.plot(center[0] + radius * np.cos(theta), center[1] + radius * np.sin(theta), **kwargs)

fig, ax = plt.subplots(figsize=(6, 6))
for i in range(len(masses)):
    traj = sim.positions[:, i, :]
    ax.plot(traj[:, 0], traj[:, 1], alpha=0.55)
    ax.scatter(traj[0, 0], traj[0, 1], s=50, marker='o')
    ax.scatter(traj[-1, 0], traj[-1, 1], s=80, marker='x')

draw_circle(ax, left_center, circle_radius, linestyle='--')
draw_circle(ax, right_center, circle_radius, linestyle='--')
ax.scatter([left_center[0], right_center[0]], [left_center[1], right_center[1]], marker='*', s=180)
ax.set_xlim(0.0, 1.0)
ax.set_ylim(0.0, 1.0)
ax.set_aspect('equal')
ax.set_title('Gaussian-mixture h-transform: one seed')
ax.set_xlabel('x')
ax.set_ylabel('y')
plt.show()



## Multi-seed check

With the parameters above I found this scheme much more stable than the two-bump cylinder target. The cell below checks seeds `0,1,...,49` and reports how often **all six particles** land inside one of the two circles.


In [ ]:

seed_stats = []
for seed in range(50):
    rng = np.random.default_rng(seed)
    sim_seed = simulate_gaussian_mixture_terminal_em(
        masses=masses,
        target_configurations=target_configurations,
        component_weights=component_weights,
        lambda_=lambda_,
        horizon=horizon,
        step_size=step_size,
        initial_positions=initial_positions,
        rng=rng,
        store_drifts=False,
    )
    stats_seed = final_circle_stats(sim_seed.positions[-1], left_center, right_center, circle_radius)
    seed_stats.append(stats_seed)

successes = sum(s['all_inside_union'] for s in seed_stats)
worst = max(s['max_distance_to_nearest_circle'] for s in seed_stats)
left_counts = sorted(set(s['left_count'] for s in seed_stats))
right_counts = sorted(set(s['right_count'] for s in seed_stats))

print(f'successes: {successes} / 50')
print('left-count values seen:', left_counts)
print('right-count values seen:', right_counts)
print('worst final nearest-circle distance:', worst)



## Notes

1. The **time step matters a lot**. For this strong conditioning, `1e-4` is reliable; `5e-4` is already noticeably worse.
2. This is still an $h$-transform: the terminal function is positive, and the drift is obtained from `2 D log u_t` just as in Section 3.1 of the paper.
3. If you are happy with a fully labelled target, plain Algorithm 1 with three targets at the left center and three at the right center also works once the step size is made equally small.
